install pydantic, google-genai, instructor, httpx, beautifulsoup4, pypdf

In [1]:
import sys
from pathlib import Path
from dotenv import load_dotenv

notebook_dir = Path().resolve()

project_root = notebook_dir.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

sys.modules.pop("connectors", None)

load_dotenv(project_root / ".env", override=True)

print("The project root has been successfully added to the Python path:")
print(f"   -> {project_root}")


The project root has been successfully added to the Python path:
   -> D:\Cods\python_project\job-ai-assist


In [2]:
from local_connectors.resume_loader import load_resume_text

resume_path = project_root / "data" / "CV_OleksiiShcherbynin.pdf"
resume_text = load_resume_text(resume_path)

print("Resume text loaded successfully.")
print("Length of resume text:", len(resume_text))
print("First 500 characters of resume text:", resume_text[:500])


Resume text loaded successfully.
Length of resume text: 3539
First 500 characters of resume text: Oleksii Shcherbynin
Bratislava, Slovakia|[redacted-phone]|[redacted-email]|linkedin.com/in/oleksii-shcherbynin-692a5a410|
github.com/OleksiiShcherbynin
Education
Slovak University of Technology (STU)Bratislava, Slovakia
Bachelor of Science in Informatics (FIIT) Sep. 2025 – Present
• Relevant Coursework:Computer Programming (C, Java, Data Structures, Algorithms), Theoretical
Foundations of Computer Science (Discrete Mathematics, Mathematical Logic, Algebra), Computer Engineering
(Architect


In [3]:
import re

def senitize_text(text: str) -> str:
    text = text.replace('\uf0b7', '-')
    text = re.sub(r'\n{3,}', '\n\n ', text) # Replace 3 or more consecutive newlines with 2 newlines followed by a space
    text = re.sub(r' {2,}', ' ', text) # '  ' -> ' '

    return text.strip()

from local_connectors.resume_loader import load_resume_text
resume_path = project_root / "data" / "CV_OleksiiShcherbynin.pdf"
raw_text = load_resume_text(resume_path)

cleaned_text = senitize_text(raw_text)

print("Raw text length:", len(raw_text))
print("Cleaned text length:", len(cleaned_text))
print("First 500 characters of resume text:", cleaned_text[:500])

Raw text length: 3539
Cleaned text length: 3539
First 500 characters of resume text: Oleksii Shcherbynin
Bratislava, Slovakia|[redacted-phone]|[redacted-email]|linkedin.com/in/oleksii-shcherbynin-692a5a410|
github.com/OleksiiShcherbynin
Education
Slovak University of Technology (STU)Bratislava, Slovakia
Bachelor of Science in Informatics (FIIT) Sep. 2025 – Present
• Relevant Coursework:Computer Programming (C, Java, Data Structures, Algorithms), Theoretical
Foundations of Computer Science (Discrete Mathematics, Mathematical Logic, Algebra), Computer Engineering
(Architect


In [4]:
from core.models import (CandidateProfile, SearchPreferences, Vacancy, ExperienceEntry, Education, LanguageSkill, MatchResult)
from core.ports import VacancySource, MessageSender, MessageReceiver

print("Models and ports imported successfully.")

Models and ports imported successfully.


In [20]:
import os
import importlib
import local_connectors.llm as llm

importlib.reload(llm)
extract = llm.extract

from core.models import CandidateProfile

if not os.getenv("GOOGLE_API_KEY"):
    print("GOOGLE_API_KEY is not set, so extract cannot be called yet.")
else:
    profile = extract(
        resume_text,
        CandidateProfile,
        instruction=(
            "Extract the candidate's profile information from the resume text and populate the CandidateProfile data model. Ensure that all relevant fields are filled accurately based on the content of the resume."
            "If there is no field, leave None or an empty list."
        ),
    )

    print(f"👤 {profile.full_name}")
    print(f"   Roll:     {profile.title}")
    print(f"   Location:  {profile.location}")
    print(f"   Experience:   {profile.years_experience} years ({profile.seniority})")
    print(f"   Tech Stack:     {profile.tech_stack}")
    print(f"   Languages:     {[(l.language_code, l.level) for l in profile.languages]}")
    print(f"   Education:   {[(e.degree, e.field) for e in profile.education]}")
    print(f"\n📋 Resume:   {(profile.summary or '')[:150]}...")

👤 Oleksii Shcherbynin
   Roll:     Software Developer
   Location:  Bratislava, Slovakia
   Experience:   0.0 years (Junior)
   Tech Stack:     ['Python', 'C (C99)', 'Java', 'SQL', 'LangChain', 'Qdrant Cloud', 'FastEmbed', 'RAG Architecture', 'NLP', 'Semantic Search', 'Streamlit', 'Git', 'Docker', 'JUnit 5', 'Gradle', 'GCC', 'REST APIs']
   Languages:     [('en', 'B1-B2'), ('sk', 'B2'), ('ru', 'Native'), ('uk', 'Native')]
   Education:   [('Bachelor of Science in Informatics (FIIT)', 'Informatics'), ('High School Diploma', None), ('Diploma in Programming and Software Development', None)]

📋 Resume:   ...


In [22]:
from core.models import SearchPreferences, WorkFormat

prefs = SearchPreferences(
    desired_roles=["Junior", "AI", "študent"],
    min_salary=0,
    work_formats=[WorkFormat.ONSITE, WorkFormat.HYBRID, WorkFormat.REMOTE],
    locations=["Bratislava", "Europe"],
    must_have=[],
    deal_breakers=["Senior", "Midle", "English C1", "English C2"],
)

print(f"Search Preferences: {prefs.desired_roles}, \nMin Salary: {prefs.min_salary}, \nWork Formats: {[wf.value for wf in prefs.work_formats]}, \nLocations: {prefs.locations}, \nMust Have: {prefs.must_have}, \nDeal Breakers: {prefs.deal_breakers}")

Search Preferences: ['Junior', 'AI', 'študent'], 
Min Salary: 0.0, 
Work Formats: ['onsite', 'hybrid', 'remote'], 
Locations: ['Bratislava', 'Europe'], 
Must Have: [], 
Deal Breakers: ['Senior', 'Midle', 'English C1', 'English C2']


----------------------------------------------

In [7]:
from local_connectors.vacancy_source import ProfesiaSource

source = ProfesiaSource(use_cache=False, start_page=1, stop_page=5)

raw_postings = source.fetch_vacancies(limit=89)

print(f"Отримано оголошень: {len(raw_postings)}")
print("\n--- перше оголошення (URL + початок тексту) ---")
first = raw_postings[0]
print(first["url"])
print(first["title"])
print(first["text"][:600])

Отримано оголошень: 89

--- перше оголошення (URL + початок тексту) ---
https://www.profesia.sk/praca/yanfeng-international-automotive-technology-slovakia/O5316680?search_id=18deb1cd-3443-4e15-9f1d-36dce944e122
Študent – AI IT Tím
Hľadanie práce Podľa regiónov Bratislavský kraj Banskobystrický kraj Žilinský kraj Trenčiansky kraj Trnavský kraj Nitriansky kraj Prešovský kraj Košický kraj Zahraničie Brigády Všetky ponuky Porovnanie mzdy Vytvorte si životopis Tip Prihlásiť S prihlásením je to lepšie Zistíte, či firma videla váš životopis, získate prehľad o histórii reakcií a na inzeráty odpoviete jednoduchšie s predvyplnenými údajmi. Rýchlo si tiež vytvoríte životopis a sprístupníte ho pre firmy. Prihlásiť sa Vytvorte si životopis Poradíme Vám? Vstup pre firmy S prihlásením je to lepšie Zistíte, či firma videla váš životopi


In [8]:
from local_connectors.llm import extract
from core.models import Vacancy

VACANCY_INSTRUCTION = (
    "Extract the structured data of the vacancy from the text of the announcement. "
    "Obviously, find out the URL of the vacancy (apply_url) - in the row 'URL: ...' on the cob."
    "Viznach language, which is written dumb (posting_language, ISO 639-1: 'sk', 'en'...). "
    "The raw_text field is left empty - please fill in the code."
    "If fields are missing — None or empty list."
)

vacancies: list[Vacancy] = []
for posting in raw_postings:
    text = posting["text"]
    v = extract(text, Vacancy, instruction=VACANCY_INSTRUCTION)
    v.raw_text = text
    vacancies.append(v)

print(f"Structured vacancies: {len(vacancies)}\n")
for v in vacancies:
    fmt = v.work_format.value if v.work_format else "—"
    lang = v.posting_language or "?"
    print(f"{(v.company or '—')[:20]:20} | {(v.role or '—')[:30]:30} | {fmt:6} | {lang}")

Structured vacancies: 89

Yanfeng Internationa | Študent – AI IT Tím            | hybrid | sk
SourceFirst Internat | Junior/Medior IT Analytik/čka  | hybrid | sk
Accace Management s. | Junior System Specialist (IT S | —      | sk
COMGUARD s.r.o.      | Junior obchodník (obchodníčka) | hybrid | sk
Letisko M.R.Štefánik | Nakladač/Nakladačka batožiny   | —      | sk
Grant Thornton Slove | Junior Auditor/Junior Auditork | hybrid | sk
Best Hotel Propertie | IT Administrator               | —      | sk
ManpowerGroup Sloven | Junior Researcher              | onsite | sk
SourceFirst Internat | IT Security Špecialista/Špecia | hybrid | sk
Raiffeisen Processin | IT Authorization Analyst (seni | hybrid | en
Aliter Technologies, | Delivery Specialist Junior     | hybrid | sk
Adifex, a. s.        | Prípravár stavieb junior       | onsite | sk
AKMV advokátska kanc | Právny/a asistent/ka - študent | onsite | sk
Kaufland Slovenská r | Brigádnik - študent (m/ž) - ob | onsite | sk
BENCONT FINANCE, a.  |

--------------------------------

In [9]:
from core.logic import rejection_reason

survivors: list[Vacancy] = []
for v in vacancies:
    reason = rejection_reason(v, prefs, profile)
    if reason:
        print(f"✗ {(v.company or '—')[:20]:20} — {reason}")
    else:
        survivors.append(v)
        print(f"✓ {(v.company or '—')[:20]:20} — Passed the filter")

print(f"\nPassed the filter:: {len(survivors)} з {len(vacancies)}")

✓ Yanfeng Internationa — Passed the filter
✗ SourceFirst Internat — no mandatory: 'študent'
✗ Accace Management s. — no mandatory: 'študent'
✗ COMGUARD s.r.o.      — no mandatory: 'študent'
✓ Letisko M.R.Štefánik — Passed the filter
✓ Grant Thornton Slove — Passed the filter
✗ Best Hotel Propertie — no mandatory: 'študent'
✓ ManpowerGroup Sloven — Passed the filter
✗ SourceFirst Internat — no mandatory: 'študent'
✗ Raiffeisen Processin — no mandatory: 'študent'
✓ Aliter Technologies, — Passed the filter
✗ Adifex, a. s.        — no mandatory: 'študent'
✓ AKMV advokátska kanc — Passed the filter
✓ Kaufland Slovenská r — Passed the filter
✗ BENCONT FINANCE, a.  — no mandatory: 'študent'
✗ Achernar Nata s.r.o. — no mandatory: 'študent'
✗ SourceFirst Internat — no mandatory: 'študent'
✓ Premedix             — Passed the filter
✗ BEAUTEX s.r.o        — no mandatory: 'študent'
✗ SWAN, a.s.           — no mandatory: 'študent'
✗ Asseco Central Europ — no mandatory: 'študent'
✗ Takenaka Europe G

In [10]:
from core.logic import missing_fields, has_enough_info

shortlist_candidates: list[tuple[Vacancy, list[str]]] = []
for v in survivors:
    gaps = missing_fields(v, prefs)
    if has_enough_info(v, prefs):
        shortlist_candidates.append((v, gaps))
        note = f"missing: {gaps}" if gaps else "full data"
        print(f"✓ {(v.company or '—')[:20]:20} — on the shortlist ({note})")
    else:
        print(f"✗ {(v.company or '—')[:20]:20} — not enough data: {gaps}")

if not survivors:
    print("\nNo vacancies survived the first filter. Relax prefs.must_have / deal_breakers if you want shortlist candidates.")

print(f"\nOn the shortlist: {len(shortlist_candidates)} out of {len(survivors)} after filter (from {len(vacancies)} raw vacancies)")


✓ Yanfeng Internationa — on the shortlist (full data)
✓ Letisko M.R.Štefánik — on the shortlist (missing: ['work format'])
✓ Grant Thornton Slove — on the shortlist (full data)
✓ ManpowerGroup Sloven — on the shortlist (full data)
✓ Aliter Technologies, — on the shortlist (full data)
✓ AKMV advokátska kanc — on the shortlist (full data)
✓ Kaufland Slovenská r — on the shortlist (missing: ['tech stack'])
✓ Premedix             — on the shortlist (full data)
✓ ARVAL COMPETENCE CEN — on the shortlist (full data)
✓ WEARMEE s. r. o.     — on the shortlist (full data)
✓ Tatra Billing, a.s.  — on the shortlist (full data)
✓ ING Hubs Slovakia    — on the shortlist (missing: ['tech stack'])
✓ ALTEP s. r. o.       — on the shortlist (full data)
✓ Trenkwalder, a. s.   — on the shortlist (missing: ['tech stack'])
✓ Bionorica Czech Repu — on the shortlist (full data)
✓ ORLEN Unipetrol Slov — on the shortlist (full data)
✓ CGI Slovensko        — on the shortlist (full data)
✓ Programko s. r. o.   — 

In [21]:
from local_connectors.llm import score_match

profile_summary = f"{profile.title}, {profile.seniority}, stack: {profile.tech_stack}"
prefs_summary = f"roles: {prefs.desired_roles}, from {prefs.min_salary}"

shortlist: list[tuple[Vacancy, MatchResult, list[str]]] = []
for v, gaps in shortlist_candidates:
    m = score_match(v.raw_text, profile_summary, prefs_summary)
    shortlist.append((v, m, gaps))

shortlist.sort(key=lambda t: t[1].score, reverse=True)

for v, m, gaps in shortlist:
    flag = f"  clarify: {gaps}" if gaps else ""
    print(f"{m.score:>3}  {(v.company or '—')[:20]:20} — {(v.role or '—')[:32]}{flag}")
    for r in m.reasons[:3]:
        print(f"       • {r}")

  ⏳ API gemini-2.5-flash: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your cu... (attempt 1/4, waiting 2с)
  ⏳ API gemini-2.5-flash: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your cu... (attempt 2/4, waiting 4с)
  ⏳ API gemini-2.5-flash: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your cu... (attempt 3/4, waiting 6с)
  ⏳ API gemini-2.5-flash: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your cu... (attempt 1/4, waiting 2с)
  ⏳ API gemini-2.5-flash: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your cu... (attempt 2/4, waiting 4с)
  ⏳ API gemini-2.5-flash: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your cu... (attempt 3/4, waiting 6с)
  ⏳ API gemini-2.5-flash: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your cu... (attempt 4/4, waiting 8с)
  🔄 Model gemini-2.5-flash unavailable, trying n